# 01 Gemini Corpus Filter

Notebook này tạo corpus pháp luật lao động bằng `gemini-3.1-flash-lite`. Không dùng keyword list thủ công; mỗi batch văn bản được phân loại bằng metadata, title và excerpt ngắn. Kết quả có cache JSONL để rerun không gọi lại Gemini.

In [1]:
from pathlib import Path
import json
import math
import os
import re
import sys
from typing import Iterator

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
from huggingface_hub import hf_hub_download
from IPython.display import Markdown, display
import pandas as pd
import pyarrow.parquet as pq

from src.data.clean_text import normalize_text
from src.data.load_dataset import DATASET_NAME
from src.data.schema import LegalDocument
from src.generation.gemini_client import BatchGeminiClient

OUTPUT_CORPUS = PROJECT_ROOT / "data/processed/labor_corpus.jsonl"
AUDIT_CACHE = PROJECT_ROOT / "reports/labor_filter_gemini.jsonl"
SUMMARY_JSON = PROJECT_ROOT / "reports/labor_filter_summary.json"

MODEL = "gemini-3.1-flash-lite"
PROMPT_VERSION = "labor_filter_v1"
GEMINI_FILTER_RPM_PER_KEY = 12
DAILY_REQUEST_BUFFER_PER_KEY = 450
BATCH_SIZE = 120
MAX_CONTENT_CHARS = 500
CONFIDENCE_THRESHOLD = 0.70

OUTPUT_CORPUS.parent.mkdir(parents=True, exist_ok=True)
AUDIT_CACHE.parent.mkdir(parents=True, exist_ok=True)
load_dotenv(PROJECT_ROOT / ".env")

C:\Users\Acer Nitro\.cache\codex-runtimes\codex-primary-runtime\dependencies\python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## 1. Load Public Dataset Metadata

Notebook đọc trực tiếp parquet của dataset gốc. Phần content được stream theo batch để không phải giữ toàn bộ corpus trong RAM.

In [2]:
def dataset_file(filename: str) -> str:
    return hf_hub_download(repo_id=DATASET_NAME, filename=filename, repo_type="dataset")


def iter_parquet_rows(parquet_file, columns: list[str] | None = None) -> Iterator[dict]:
    for batch in parquet_file.iter_batches(batch_size=512, columns=columns):
        yield from batch.to_pylist()


metadata_file = pq.ParquetFile(dataset_file("data/metadata.parquet"))
content_file = pq.ParquetFile(dataset_file("data/content.parquet"))
metadata_by_id = {str(row["id"]): row for row in iter_parquet_rows(metadata_file)}

print(f"metadata rows={metadata_file.metadata.num_rows}")
print(f"content rows={content_file.metadata.num_rows}")
print(f"metadata matched by id={len(metadata_by_id)}")

metadata rows=153420
content rows=178665
metadata matched by id=153420


In [3]:
def make_document(metadata_row: dict, content_html: str) -> LegalDocument:
    data = dict(metadata_row)
    data["id"] = str(data["id"])
    data["content_text"] = normalize_text(content_html)
    return LegalDocument(**data)


def iter_source_documents() -> Iterator[LegalDocument]:
    seen_ids = set()
    for content_row in iter_parquet_rows(content_file, columns=["id", "content_html"]):
        doc_id = str(content_row["id"])
        if doc_id in seen_ids:
            continue
        seen_ids.add(doc_id)
        metadata_row = metadata_by_id.get(doc_id)
        if metadata_row is None:
            continue
        yield make_document(metadata_row, content_row.get("content_html", ""))


def compact_document(document: LegalDocument) -> dict:
    return {
        "id": document.id,
        "title": document.title,
        "so_ky_hieu": document.so_ky_hieu,
        "ngay_ban_hanh": document.ngay_ban_hanh,
        "loai_van_ban": document.loai_van_ban,
        "ngay_co_hieu_luc": document.ngay_co_hieu_luc,
        "ngay_het_hieu_luc": document.ngay_het_hieu_luc,
        "nganh": document.nganh,
        "linh_vuc": document.linh_vuc,
        "co_quan_ban_hanh": document.co_quan_ban_hanh,
        "pham_vi": document.pham_vi,
        "tinh_trang_hieu_luc": document.tinh_trang_hieu_luc,
        "content_text": document.content_text,
        "source_url": document.source_url,
    }


preview_rows = []
for document in iter_source_documents():
    preview_rows.append(compact_document(document))
    if len(preview_rows) == 10:
        break
pd.DataFrame(preview_rows)[["id", "title", "loai_van_ban", "nganh", "linh_vuc"]]

,id,title,loai_van_ban,nganh,linh_vuc
0,4260,Về việc chuyển giao nhiệm vụ quản lý Nhà nước ...,Quyết định,NaN,None
1,4266,"Về việc tăng cường công tác quản lý nhà, đất v...",Chỉ thị,NaN,None
2,4262,Về việc tăng cường quản lý đất lâm nghiệp,Chỉ thị,NaN,None
3,4264,Về việc kinh doanh mặt hàng rượu,Chỉ thị,NaN,None
4,4281,Quản lý thống nhất việc thu phí vào cổng tham ...,Quyết định,Tài chính,None
5,4265,Về việc thực hiện Quyết định số 53/1999/QĐ-TTg...,Chỉ thị,NaN,None
6,4299,Về việc quy định chế độ công tác phí Hội nghị ...,Quyết định,Tài chính,None
7,4273,Về việc chuyển chức năng QLNN về Thể dục thể t...,Quyết định,Văn hóa - Thông tin,None
8,4274,Về việc giao chỉ tiêu kế hoạch động viên và hu...,Quyết định,Quốc phòng,None
9,4286,Về việc ban hành Qui chế hoạt động của Hội đồn...,Quyết định,Giáo dục và đào tạo Quốc phòng Tài chính,None


## 2. Gemini Classifier

Mỗi request phân loại tối đa 60 văn bản. Limiter là `12 RPM/key`, thấp hơn quota `15 RPM/key`; cache giúp notebook resume nếu bị ngắt hoặc hết quota.

In [4]:
def gemini_key_count() -> int:
    raw_values = [
        os.getenv("GOOGLE_API_KEY") or "",
        os.getenv("GEMINI_API_KEY") or "",
        os.getenv("GEMINI_API_KEYS") or "",
    ]
    keys = []
    for raw_value in raw_values:
        for key in raw_value.split(","):
            key = key.strip()
            if key and key not in keys:
                keys.append(key)
    return len(keys)


def load_audit_cache() -> dict[str, dict]:
    if not AUDIT_CACHE.exists():
        return {}
    cache = {}
    with AUDIT_CACHE.open(encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            if row.get("prompt_version") == PROMPT_VERSION:
                cache[str(row["id"])] = row
    return cache


def append_audit_rows(rows: list[dict]) -> None:
    with AUDIT_CACHE.open("a", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def document_payload(document: LegalDocument) -> dict:
    return {
        "id": document.id,
        "title": document.title,
        "nganh": document.nganh,
        "linh_vuc": document.linh_vuc,
        "loai_van_ban": document.loai_van_ban,
        "co_quan_ban_hanh": document.co_quan_ban_hanh,
        "thong_tin_ap_dung": document.thong_tin_ap_dung,
        "content_excerpt": normalize_text(document.content_text)[:MAX_CONTENT_CHARS],
    }


def build_filter_prompt(batch: list[LegalDocument]) -> str:
    payload = [document_payload(document) for document in batch]
    return f"""
Bạn là bộ phân loại corpus cho dự án Vietnamese labor-law RAG.
Nhiệm vụ: xác định từng văn bản có thuộc pháp luật lao động hay không.

Accept nếu trọng tâm văn bản điều chỉnh quan hệ lao động, việc làm, tiền lương,
hợp đồng lao động, bảo hiểm xã hội/bảo hiểm thất nghiệp trong bối cảnh lao động,
công đoàn, an toàn vệ sinh lao động, đào tạo nghề, tranh chấp lao động.
Reject nếu chỉ nhắc lao động/cán bộ/công chức phụ trợ trong văn bản thuộc đất đai,
thuế, đầu tư, hình sự, y tế, giáo dục, tài chính hoặc thủ tục hành chính khác.

Trả về JSON duy nhất, không markdown:
{{"results":[{{"id":"...","is_labor_law":true,"confidence":0.0,"reason":"ngắn gọn"}}]}}

Documents:
{json.dumps(payload, ensure_ascii=False)}
""".strip()


def parse_json_response(text: str) -> dict:
    cleaned = re.sub(r"^```json|```$", "", text.strip(), flags=re.MULTILINE).strip()
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start == -1 or end == -1:
        raise ValueError("Gemini response does not contain JSON")
    return json.loads(cleaned[start : end + 1])


def classify_batch(client: BatchGeminiClient, batch: list[LegalDocument]) -> list[dict]:
    prompt = build_filter_prompt(batch)
    parsed, last_error = None, None
    for _ in range(2):
        try:
            parsed = parse_json_response(client.generate(prompt))
            break
        except Exception as exc:
            last_error = exc
    if parsed is None:
        return [{
            "id": document.id,
            "title": document.title,
            "is_labor_law": False,
            "confidence": 0.0,
            "reason": f"classification_error: {last_error}",
            "model": MODEL,
            "prompt_version": PROMPT_VERSION,
            "cached": False,
        } for document in batch]
    by_id = {str(row.get("id")): row for row in parsed.get("results", [])}
    rows = []
    for document in batch:
        result = by_id.get(document.id, {})
        rows.append({
            "id": document.id,
            "title": document.title,
            "is_labor_law": bool(result.get("is_labor_law")),
            "confidence": float(result.get("confidence") or 0.0),
            "reason": str(result.get("reason") or ""),
            "model": MODEL,
            "prompt_version": PROMPT_VERSION,
            "cached": False,
        })
    return rows

## 3. Run Gemini Filter

Notebook tự tính safe daily request budget từ số key hiện có. Với quota `500 RPD/key`, mặc định chỉ dùng `450 request/key` để còn buffer.

In [5]:
cache = load_audit_cache()
key_count = gemini_key_count()
safe_request_budget = key_count * DAILY_REQUEST_BUFFER_PER_KEY
pending_batch, new_rows = [], []
request_count = 0
client = None

for document in iter_source_documents():
    if document.id in cache:
        continue
    pending_batch.append(document)
    if len(pending_batch) < BATCH_SIZE:
        continue
    if request_count >= safe_request_budget:
        break
    if client is None:
        client = BatchGeminiClient(model=MODEL, rpm_limit=GEMINI_FILTER_RPM_PER_KEY)
    rows = classify_batch(client, pending_batch)
    append_audit_rows(rows)
    cache.update({row["id"]: row for row in rows})
    new_rows.extend(rows)
    request_count += 1
    pending_batch = []
    if request_count == 1 or request_count % 25 == 0:
        print(f"requests={request_count}/{safe_request_budget}, cached={len(cache)}")

if pending_batch and request_count < safe_request_budget:
    if client is None:
        client = BatchGeminiClient(model=MODEL, rpm_limit=GEMINI_FILTER_RPM_PER_KEY)
    rows = classify_batch(client, pending_batch)
    append_audit_rows(rows)
    cache.update({row["id"]: row for row in rows})
    new_rows.extend(rows)
    request_count += 1

print(f"keys={key_count}")
print(f"new_requests={request_count}")
print(f"new_decisions={len(new_rows)}")
print(f"cached_decisions={len(cache)}")

keys=6
new_requests=0
new_decisions=0
cached_decisions=146857


## 4. Write Filtered Corpus

Corpus chỉ được ghi sau khi cache đã có decision cho toàn bộ source documents. Nếu quota hết giữa chừng, rerun notebook để tiếp tục.

In [6]:
accepted_ids = set()
total_documents = 0
missing_decisions = 0
for document in iter_source_documents():
    total_documents += 1
    row = cache.get(document.id)
    if row is None:
        missing_decisions += 1
        continue
    if row.get("is_labor_law") and float(row.get("confidence") or 0.0) >= CONFIDENCE_THRESHOLD:
        accepted_ids.add(document.id)

completed = missing_decisions == 0
accepted_count = len(accepted_ids)
if completed:
    with OUTPUT_CORPUS.open("w", encoding="utf-8") as f:
        for document in iter_source_documents():
            if document.id in accepted_ids:
                f.write(json.dumps(compact_document(document), ensure_ascii=False) + "\n")
summary = {
    "model": MODEL,
    "prompt_version": PROMPT_VERSION,
    "rpm_per_key": GEMINI_FILTER_RPM_PER_KEY,
    "daily_request_buffer_per_key": DAILY_REQUEST_BUFFER_PER_KEY,
    "batch_size": BATCH_SIZE,
    "max_content_chars": MAX_CONTENT_CHARS,
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "total_documents": total_documents,
    "cached_decisions": len(cache),
    "new_decisions": len(new_rows),
    "missing_decisions": missing_decisions,
    "accepted": accepted_count,
    "completed": completed,
    "output_written": completed,
}
SUMMARY_JSON.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

pd.DataFrame([summary])

,model,prompt_version,rpm_per_key,daily_request_buffer_per_key,batch_size,max_content_chars,confidence_threshold,total_documents,cached_decisions,new_decisions,missing_decisions,accepted,completed,output_written
0,gemini-3.1-flash-lite,labor_filter_v1,12,450,120,500,0.7,146857,146857,0,0,17379,True,True


In [7]:
if OUTPUT_CORPUS.exists():
    accepted_preview = []
    with OUTPUT_CORPUS.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                accepted_preview.append(json.loads(line))
            if len(accepted_preview) == 20:
                break
    display(pd.DataFrame(accepted_preview)[["id", "title", "loai_van_ban", "nganh", "linh_vuc"]])
else:
    display(Markdown("`labor_corpus.jsonl` chưa được ghi vì filter chưa hoàn tất. Rerun notebook để tiếp tục từ cache."))

,id,title,loai_van_ban,nganh,linh_vuc
0,4299,Về việc quy định chế độ công tác phí Hội nghị ...,Quyết định,Tài chính,None
1,4284,Về việc tăng cường quản lý nhà nước trong lĩnh...,Chỉ thị,NaN,None
2,4295,Về việc quy định tạm thời mức bồi dưỡng tập lu...,Quyết định,Tài chính Văn hóa Thể thao và Du lịch,None
3,4276,"Về việc điều chỉnh danh mục các xã, phường, th...",Quyết định,NaN,None
4,74,Sắc lệnh ấn định lương và phụ cấp cho các công...,Sắc lệnh,NaN,None
5,83,Sắc lệnh ấn định lương và phụ cấp cho các cấp ...,Sắc lệnh,NaN,None
6,91,Sắc lệnh quy định chế độ công nhân giúp việc C...,Sắc lệnh,NaN,None
7,36,Sắc lệnh sửa đổi Sắc lệnh số 200-SL ngày 8-7-1...,Sắc lệnh,NaN,None
8,4204,Ban hành quy định phân cấp quản lý công tác Tổ...,Quyết định,NaN,None
9,96,Sắc lệnh ban hành Quy chế Công chức,Sắc lệnh,NaN,None
